In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nlp-getting-started/sample_submission.csv
/kaggle/input/competitions/nlp-getting-started/train.csv
/kaggle/input/competitions/nlp-getting-started/test.csv


In [2]:
import numpy as np
import pandas as pd

In [3]:
train = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")
test = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")

In [4]:
print(train.shape)
print(test.shape)

(7613, 5)
(3263, 4)


In [5]:
display(train.head())
display(test.head())

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [6]:
print(train.info())
print("\nMissing values:")
print(train.isna().sum())

print("\nTarget distribution:")
print(train["target"].value_counts())
print(train["target"].value_counts(normalize=True))

print("\nDuplicate tweets:")
print(train["text"].duplicated().sum())

print("\nSample tweets:")
for i in range(5):
    print(f"\n[{train.loc[i, 'target']}] {train.loc[i, 'text']}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB
None

Missing values:
id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

Target distribution:
target
0    4342
1    3271
Name: count, dtype: int64
target
0    0.57034
1    0.42966
Name: proportion, dtype: float64

Duplicate tweets:
110

Sample tweets:

[1] Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all

[1] Forest fire near La Ronge Sask. Canada

[1] All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected

[1] 13,000 people

In [7]:
print("Unique keywords:", train["keyword"].nunique())
print("Unique locations:", train["location"].nunique())

print("\nTop keywords:")
print(train["keyword"].value_counts().head(15))

print("\nTarget by keyword:")
print(
    train.groupby("keyword")["target"]
    .agg(["count", "mean"])
    .sort_values("count", ascending=False)
    .head(15)
)

Unique keywords: 221
Unique locations: 3341

Top keywords:
keyword
fatalities     45
deluge         42
armageddon     42
damage         41
body%20bags    41
harm           41
sinking        41
evacuate       40
outbreak       40
fear           40
siren          40
windstorm      40
collided       40
twister        40
hellfire       39
Name: count, dtype: int64

Target by keyword:
             count      mean
keyword                     
fatalities      45  0.577778
deluge          42  0.142857
armageddon      42  0.119048
damage          41  0.463415
body%20bags     41  0.024390
harm            41  0.097561
sinking         41  0.195122
evacuate        40  0.625000
outbreak        40  0.975000
fear            40  0.125000
siren           40  0.125000
windstorm       40  0.400000
collided        40  0.575000
twister         40  0.125000
hellfire        39  0.179487


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    train["text"],
    train["target"],
    test_size=0.2,
    random_state=42,
    stratify=train["target"]
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

Train: (6090,)
Validation: (1523,)


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_valid_tfidf = tfidf.transform(X_valid)

print("Train matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_valid_tfidf.shape)

model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

valid_pred = model.predict(X_valid_tfidf)

f1 = f1_score(y_valid, valid_pred)

print("Validation F1:", f1)
print("\nClassification report:")
print(classification_report(y_valid, valid_pred))

Train matrix: (6090, 14759)
Validation matrix: (1523, 14759)
Validation F1: 0.7594728171334432

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       869
           1       0.82      0.70      0.76       654

    accuracy                           0.81      1523
   macro avg       0.81      0.80      0.80      1523
weighted avg       0.81      0.81      0.81      1523



In [10]:
import numpy as np

feature_names = np.array(tfidf.get_feature_names_out())
coefficients = model.coef_[0]

top_positive = np.argsort(coefficients)[-20:][::-1]
top_negative = np.argsort(coefficients)[:20]

print("Strongest disaster indicators:\n")

for i in top_positive:
    print(
        f"{feature_names[i]:30s} {coefficients[i]:.3f}"
    )

print("\nStrongest non-disaster indicators:\n")

for i in top_negative:
    print(
        f"{feature_names[i]:30s} {coefficients[i]:.3f}"
    )

Strongest disaster indicators:

in                             3.511
hiroshima                      2.950
fires                          2.486
california                     2.386
fire                           2.356
storm                          2.310
train                          2.072
killed                         2.010
wildfire                       1.997
police                         1.946
suicide                        1.941
near                           1.863
bombing                        1.845
disaster                       1.841
buildings                      1.826
floods                         1.816
at                             1.763
mass                           1.758
massacre                       1.747
accident                       1.722

Strongest non-disaster indicators:

you                            -3.887
my                             -3.123
new                            -2.080
love                           -1.573
full                           -1.559
b

In [11]:
from sklearn.pipeline import FeatureUnion

word_tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True
)

combined_tfidf = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])

X_train_combined = combined_tfidf.fit_transform(X_train)
X_valid_combined = combined_tfidf.transform(X_valid)

print("Train shape:", X_train_combined.shape)
print("Validation shape:", X_valid_combined.shape)

model_combined = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

model_combined.fit(
    X_train_combined,
    y_train
)

valid_pred_combined = model_combined.predict(
    X_valid_combined
)

print(
    "Combined F1:",
    f1_score(y_valid, valid_pred_combined)
)

Train shape: (6090, 118452)
Validation shape: (1523, 118452)
Combined F1: 0.7802108678021087


In [12]:
!pip install -q sentence-transformers

In [13]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

print("Using:", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using: cuda


In [14]:
X_train_emb = embedding_model.encode(
    X_train.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_valid_emb = embedding_model.encode(
    X_valid.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Train:", X_train_emb.shape)
print("Validation:", X_valid_emb.shape)

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Train: (6090, 384)
Validation: (1523, 384)


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

embedding_clf = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

embedding_clf.fit(
    X_train_emb,
    y_train
)

emb_pred = embedding_clf.predict(X_valid_emb)

emb_f1 = f1_score(y_valid, emb_pred)

print("Embedding F1:", emb_f1)
print("\nClassification report:")
print(
    classification_report(
        y_valid,
        emb_pred
    )
)

Embedding F1: 0.8006354249404289

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       869
           1       0.83      0.77      0.80       654

    accuracy                           0.84      1523
   macro avg       0.83      0.83      0.83      1523
weighted avg       0.84      0.84      0.83      1523



In [16]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Transformers: 5.0.0
PyTorch: 2.10.0+cu128
GPU: Tesla T4


In [17]:
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

train_df = pd.DataFrame({
    "text": X_train.values,
    "label": y_train.values
})

valid_df = pd.DataFrame({
    "text": X_valid.values,
    "label": y_valid.values
})

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df, preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)

print(train_ds)
print(valid_ds)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6090
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1523
})


In [18]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print(model.config)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "vocab_size": 30522
}



In [19]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    predictions = np.argmax(logits, axis=-1)
    
    return {
        "f1": f1_score(labels, predictions),
        "accuracy": accuracy_score(labels, predictions)
    }

In [20]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_disaster",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [22]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,No log,0.755435,0.812749,0.845699
2,No log,0.786739,0.811550,0.837163
3,0.742826,0.792470,0.816680,0.847012


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=573, training_loss=0.7248029326061096, metrics={'train_runtime': 150.2377, 'train_samples_per_second': 121.607, 'train_steps_per_second': 3.814, 'total_flos': 605044843361280.0, 'train_loss': 0.7248029326061096, 'epoch': 3.0})

In [23]:
results = trainer.evaluate()

print("Validation F1:", results["eval_f1"])
print("Validation Accuracy:", results["eval_accuracy"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation F1: 0.8166797797010228
Validation Accuracy: 0.8470124753775443


In [24]:
test = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")

test_ds = Dataset.from_pandas(
    test[["text"]],
    preserve_index=False
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

In [25]:
predictions = trainer.predict(test_ds)

test_pred = np.argmax(
    predictions.predictions,
    axis=-1
)

print(test_pred[:20])
print("Predicted disasters:", test_pred.sum())

[1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0]
Predicted disasters: 1265


In [26]:
submission = pd.DataFrame({
    "id": test["id"],
    "target": test_pred
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
